# Phase 2 — full protocol-locked experiment

This notebook reads the CUDA microbatch selected by notebook 01, runs the validation-only smoke check, locks the protocol, and executes the 190-run label-efficiency matrix. It saves every checkpoint and result to Google Drive.

Before running: choose **Runtime → Change runtime type → GPU**. Prefer the same GPU class used for the benchmark when it is available. Keep this browser tab connected; pay-as-you-go units alone do not provide guaranteed background execution.

In [ ]:
import shutil
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Change the Colab runtime type to GPU, then rerun.")
subprocess.run(["nvidia-smi"], check=True)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
if shutil.which("colab"):
    print("\nCurrent Colab balance and burn rate, when exposed by this runtime:")
    subprocess.run(["colab", "usage"], check=False)

## Restore the project and persistent state

The bundle should already be in Drive from notebook 01. If it is missing, this cell asks for the zip once. The project code is extracted to fast local storage while `results/`, model downloads, and the frozen representation cache remain in Drive.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive, files

drive.mount("/content/drive")
PERSISTENT = Path("/content/drive/MyDrive/esm2-phase2-colab")
PERSISTENT.mkdir(parents=True, exist_ok=True)
BUNDLE_NAME = "esm2-mutation-effects-colab.zip"
BUNDLE = PERSISTENT / BUNDLE_NAME

if not BUNDLE.exists():
    sidebar_copy = Path("/content") / BUNDLE_NAME
    if sidebar_copy.exists():
        uploaded_bundle = sidebar_copy
    else:
        print(f"Choose {BUNDLE_NAME} in the upload dialog.")
        uploaded = files.upload()
        matches = [Path("/content") / name for name in uploaded if name.endswith(".zip")]
        if len(matches) != 1:
            raise RuntimeError("Upload exactly one project zip bundle.")
        uploaded_bundle = matches[0]
    shutil.copy2(uploaded_bundle, BUNDLE)
else:
    print(f"Reusing bundle from {BUNDLE}")

PROJECT = Path("/content/esm2-mutation-effects")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall("/content")
if not (PROJECT / "pyproject.toml").exists():
    raise RuntimeError("The uploaded zip is not the expected Colab project bundle.")

def persistent_directory(local_path, drive_path):
    drive_path.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink() or local_path.is_file():
        local_path.unlink()
    elif local_path.exists():
        shutil.rmtree(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.symlink_to(drive_path, target_is_directory=True)

persistent_directory(PROJECT / "results", PERSISTENT / "results")
persistent_directory(PROJECT / "data" / "processed", PERSISTENT / "processed")
os.environ["TORCH_HOME"] = str(PERSISTENT / "torch-cache")
Path(os.environ["TORCH_HOME"]).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print(f"Project ready at {PROJECT}")
print(f"Persistent artifacts: {PERSISTENT}")

## Load the pre-test CUDA choice

Normally leave `MICROBATCH_OVERRIDE` as `None`. An override is only for recovering when notebook 01 was run elsewhere and its summary was not copied. Once Phase 2 starts, never change this value.

In [ ]:
import json

MICROBATCH_OVERRIDE = None
benchmark_path = PROJECT / "results" / "phase2_cuda_microbatch_benchmark" / "benchmark_summary.json"
if MICROBATCH_OVERRIDE is None:
    if not benchmark_path.exists():
        raise FileNotFoundError("CUDA benchmark missing. Run 01_cuda_benchmark.ipynb first.")
    benchmark = json.loads(benchmark_path.read_text())
    MICRO_BATCH = benchmark["fastest_completed_microbatch"]
else:
    MICRO_BATCH = int(MICROBATCH_OVERRIDE)
if MICRO_BATCH not in (4, 8, 16):
    raise ValueError(f"Unexpected microbatch: {MICRO_BATCH}")
print(f"Locked candidate for this Colab run: microbatch {MICRO_BATCH}, effective batch 16 lists")

## Validate before the long run

This verifies the assay, all automated tests, the required zero-shot predictions, and the exact commands the orchestrator will issue.

In [ ]:
subprocess.run([sys.executable, "scripts/validate_dataset.py"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT, check=True)
zero_shot = PROJECT / "results" / "esm2_masked_marginal.csv"
if not zero_shot.exists():
    subprocess.run(
        [sys.executable, "scripts/score_zero_shot.py", "--device", "cuda", "--batch-size", "32"],
        cwd=PROJECT,
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "scripts/run_phase2.py",
        "--device",
        "cuda",
        "--list-microbatch-size",
        str(MICRO_BATCH),
        "--dry-run",
    ],
    cwd=PROJECT,
    check=True,
)

## Run or resume Phase 2

This is the long-running cell. Output is shown here and simultaneously appended to `MyDrive/esm2-phase2-colab/phase2_colab.log`. If the runtime is lost, rerun this notebook from the top: completed jobs are skipped and the current job resumes from its compact `last.pt` checkpoint.

In [ ]:
from datetime import datetime, timezone

command = [
    sys.executable,
    "scripts/run_phase2.py",
    "--device",
    "cuda",
    "--list-microbatch-size",
    str(MICRO_BATCH),
]
log_path = PERSISTENT / "phase2_colab.log"
with log_path.open("a", buffering=1) as log:
    marker = f"\n=== Colab invocation {datetime.now(timezone.utc).isoformat()} ===\n"
    print(marker, end="")
    log.write(marker)
    process = subprocess.Popen(
        command,
        cwd=PROJECT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f"Phase 2 stopped with code {return_code}. State is retained; inspect {log_path} and rerun from the top."
    )
print(f"Phase 2 finished successfully. Log: {log_path}")

## Inspect the completed result

This cell only reports values already written by the experiment; it does not invent or recompute missing results.

In [ ]:
import pandas as pd

summary_path = PROJECT / "results" / "phase2_label_efficiency" / "phase2_summary.json"
if not summary_path.exists():
    raise FileNotFoundError("Phase 2 summary is not present yet. The long-running cell has not completed.")
summary = json.loads(summary_path.read_text())
print(
    f"Completed supervised runs: {summary['completed_supervised_runs']}/"
    f"{summary['expected_supervised_runs']}"
)
print(f"Matrix complete: {summary['matrix_complete']}")
display(pd.DataFrame(summary["aggregate_summary"]))
print(f"Summary saved at {summary_path}")